In [ ]:
import os
os.chdir(os.environ.get("PROJECT_ROOT", "../.."))

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

# Configuration
actives_base = Path(os.environ.get("ACTIVES_BASE", "actives/domain"))
num_experts = 32
max_workers = 64

models = {
    "290m-seed3407-32e": "Baseline",
    "290m-dro-activation-beta0.999-eta0.001-alpha1.0-norml2_fine_grained_32e": "DRMoET"
}

checkpoint = 16000
domains = ["code", "math", "wiki", "web"]
layers = [2, 3, 4, 5, 6, 7, 8, 9]

# Paper style
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
})

output_dir = Path("figures/expert-specialization-32e")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Actives base: {actives_base}")
print(f"Models: {list(models.keys())}")
print(f"Domains: {domains}")
print(f"Layers: {layers}")

In [ ]:
def count_experts_in_file(pt_file: Path) -> np.ndarray:
    """Count expert selections in a single trace file."""
    counts = np.zeros(num_experts, dtype=np.float64)
    try:
        values, indices = torch.load(pt_file, map_location="cpu", weights_only=True)
        for idx in indices.flatten().numpy():
            if idx < num_experts:
                counts[idx] += 1
    except:
        pass
    return counts


def load_domain_layer_counts(model_name: str, domain: str, layer: int) -> np.ndarray:
    """Load and aggregate expert counts for a specific domain and layer."""
    layer_dir = actives_base / model_name / str(checkpoint) / domain / str(layer)
    if not layer_dir.exists():
        print(f"    Not found: {layer_dir}")
        return np.zeros(num_experts)
    
    pt_files = list(layer_dir.glob("*.pt"))
    if not pt_files:
        return np.zeros(num_experts)
    
    counts = np.zeros(num_experts, dtype=np.float64)
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(count_experts_in_file, f) for f in pt_files]
        for future in futures:
            counts += future.result()
    
    return counts


def load_domain_counts(model_name: str, domain: str) -> dict:
    """Load expert counts for all layers for a domain."""
    result = {}
    total = np.zeros(num_experts, dtype=np.float64)
    
    for layer in layers:
        counts = load_domain_layer_counts(model_name, domain, layer)
        result[layer] = counts
        total += counts
    
    result['total'] = total
    return result

In [ ]:
# Load all data
all_data = {}

for model_name, label in models.items():
    print(f"\nLoading {label}...")
    all_data[model_name] = {}
    
    for domain in tqdm(domains, desc="Domains"):
        all_data[model_name][domain] = load_domain_counts(model_name, domain)

print("\nData loading complete!")

In [ ]:
# Compute specialization metrics
# For each expert, compute what fraction of its tokens come from each domain

def compute_expert_domain_distribution(model_name: str) -> np.ndarray:
    """Compute distribution of domains for each expert.
    Returns matrix: (num_experts, num_domains) where each row sums to 1.
    """
    # First, get total counts per expert per domain (aggregated across layers)
    expert_domain_counts = np.zeros((num_experts, len(domains)))
    
    for d_idx, domain in enumerate(domains):
        total_counts = all_data[model_name][domain]['total']
        expert_domain_counts[:, d_idx] = total_counts
    
    # Normalize each expert's distribution across domains
    row_sums = expert_domain_counts.sum(axis=1, keepdims=True)
    expert_domain_dist = expert_domain_counts / (row_sums + 1e-10)
    
    return expert_domain_dist, expert_domain_counts


def compute_domain_expert_distribution(model_name: str) -> np.ndarray:
    """Compute distribution of experts for each domain.
    Returns matrix: (num_domains, num_experts) where each row sums to 1.
    """
    domain_expert_counts = np.zeros((len(domains), num_experts))
    
    for d_idx, domain in enumerate(domains):
        total_counts = all_data[model_name][domain]['total']
        domain_expert_counts[d_idx, :] = total_counts
    
    # Normalize each domain's distribution across experts
    row_sums = domain_expert_counts.sum(axis=1, keepdims=True)
    domain_expert_dist = domain_expert_counts / (row_sums + 1e-10)
    
    return domain_expert_dist, domain_expert_counts


# Compute for both models
specialization = {}
for model_name, label in models.items():
    expert_domain_dist, expert_domain_counts = compute_expert_domain_distribution(model_name)
    domain_expert_dist, domain_expert_counts = compute_domain_expert_distribution(model_name)
    
    specialization[model_name] = {
        'expert_domain_dist': expert_domain_dist,
        'expert_domain_counts': expert_domain_counts,
        'domain_expert_dist': domain_expert_dist,
        'domain_expert_counts': domain_expert_counts,
    }

print("Specialization metrics computed!")

In [ ]:
# Plot 1: Expert-Domain Heatmap (which domains each expert prefers)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (model_name, label) in zip(axes, models.items()):
    data = specialization[model_name]['expert_domain_dist']
    
    im = ax.imshow(data.T, aspect='auto', cmap='YlOrRd')
    ax.set_xlabel('Expert Index')
    ax.set_ylabel('Domain')
    ax.set_yticks(range(len(domains)))
    ax.set_yticklabels(domains)
    ax.set_title(f'{label}: Expert Domain Preference')
    plt.colorbar(im, ax=ax, label='Fraction of Expert\'s Tokens')

plt.suptitle('Expert Specialization by Domain', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'expert_domain_heatmap.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'expert_domain_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Domain-Expert Heatmap (which experts each domain uses)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (model_name, label) in zip(axes, models.items()):
    data = specialization[model_name]['domain_expert_dist']
    
    im = ax.imshow(data, aspect='auto', cmap='YlOrRd')
    ax.set_xlabel('Expert Index')
    ax.set_ylabel('Domain')
    ax.set_yticks(range(len(domains)))
    ax.set_yticklabels(domains)
    ax.set_title(f'{label}: Domain Expert Usage')
    # Add uniform line
    ax.axhline(-0.5, color='gray', linestyle='--', alpha=0.5)
    plt.colorbar(im, ax=ax, label='Fraction of Domain\'s Tokens')

plt.suptitle('Expert Usage by Domain (uniform = 1/32 = 3.125%)', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'domain_expert_heatmap.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'domain_expert_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Specialization Score per Expert
# Measure how specialized each expert is (high = focuses on specific domains)
# Using entropy: low entropy = high specialization

def compute_specialization_score(dist: np.ndarray) -> np.ndarray:
    """Compute specialization score (1 - normalized entropy)."""
    # Entropy for each expert
    entropy = -np.sum(dist * np.log(dist + 1e-10), axis=1)
    max_entropy = np.log(len(domains))
    # Specialization = 1 - normalized entropy (higher = more specialized)
    specialization_score = 1 - (entropy / max_entropy)
    return specialization_score

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(num_experts)
width = 0.35

colors = ['#2E86AB', '#A23B72']
for i, (model_name, label) in enumerate(models.items()):
    dist = specialization[model_name]['expert_domain_dist']
    scores = compute_specialization_score(dist)
    ax.bar(x + i*width - width/2, scores, width, label=label, color=colors[i], alpha=0.8)

ax.set_xlabel('Expert Index')
ax.set_ylabel('Specialization Score (higher = more specialized)')
ax.set_title('Expert Specialization Score by Model')
ax.legend()
ax.set_ylim(0, 1)
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'specialization_scores.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'specialization_scores.png', bbox_inches='tight')
plt.show()

# Print average scores
for model_name, label in models.items():
    dist = specialization[model_name]['expert_domain_dist']
    scores = compute_specialization_score(dist)
    print(f"{label}: avg specialization = {scores.mean():.3f}, std = {scores.std():.3f}")

In [ ]:
# Plot 4: Top experts for each domain
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, domain in zip(axes, domains):
    x = np.arange(num_experts)
    width = 0.35
    
    for i, (model_name, label) in enumerate(models.items()):
        d_idx = domains.index(domain)
        dist = specialization[model_name]['domain_expert_dist'][d_idx] * 100  # percentage
        ax.bar(x + i*width - width/2, dist, width, label=label, 
               color=['#2E86AB', '#A23B72'][i], alpha=0.8)
    
    ax.axhline(100/num_experts, color='red', linestyle='--', alpha=0.5, 
               label=f'Uniform ({100/num_experts:.1f}%)')
    ax.set_xlabel('Expert Index')
    ax.set_ylabel('Token Share (%)')
    ax.set_title(f'Domain: {domain.upper()}')
    ax.legend(loc='upper right', fontsize=9)

plt.suptitle('Expert Usage Distribution by Domain', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'domain_expert_bars.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'domain_expert_bars.png', bbox_inches='tight')
plt.show()

In [ ]:
# Plot 5: Specialization by Layer
# Does specialization differ across layers?

def compute_layer_specialization(model_name: str, layer: int) -> float:
    """Compute average specialization score for a layer."""
    # Get counts per expert per domain for this layer
    expert_domain_counts = np.zeros((num_experts, len(domains)))
    for d_idx, domain in enumerate(domains):
        layer_counts = all_data[model_name][domain].get(layer, np.zeros(num_experts))
        expert_domain_counts[:, d_idx] = layer_counts
    
    # Normalize
    row_sums = expert_domain_counts.sum(axis=1, keepdims=True)
    expert_domain_dist = expert_domain_counts / (row_sums + 1e-10)
    
    # Compute specialization scores
    scores = compute_specialization_score(expert_domain_dist)
    return scores.mean()

fig, ax = plt.subplots(figsize=(10, 6))

for model_name, label in models.items():
    layer_scores = [compute_layer_specialization(model_name, layer) for layer in layers]
    ax.plot(layers, layer_scores, 'o-', label=label, linewidth=2, markersize=8)

ax.set_xlabel('Layer')
ax.set_ylabel('Average Specialization Score')
ax.set_title('Expert Specialization by Layer')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'specialization_by_layer.pdf', bbox_inches='tight')
plt.savefig(output_dir / 'specialization_by_layer.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
print("="*60)
print("EXPERT SPECIALIZATION SUMMARY")
print("="*60)

for model_name, label in models.items():
    print(f"\n{label}:")
    
    dist = specialization[model_name]['expert_domain_dist']
    scores = compute_specialization_score(dist)
    
    print(f"  Average specialization: {scores.mean():.3f}")
    print(f"  Most specialized expert: {scores.argmax()} (score={scores.max():.3f})")
    print(f"  Least specialized expert: {scores.argmin()} (score={scores.min():.3f})")
    
    # Find domain preferences for most specialized expert
    most_spec = scores.argmax()
    prefs = dist[most_spec]
    top_domain = domains[prefs.argmax()]
    print(f"  Expert {most_spec} prefers: {top_domain} ({prefs.max()*100:.1f}%)")

print("\n" + "="*60)

# Save results
results = {
    'domains': domains,
    'num_experts': num_experts,
    'checkpoint': checkpoint,
}

for model_name, label in models.items():
    dist = specialization[model_name]['expert_domain_dist']
    scores = compute_specialization_score(dist)
    results[label.lower().replace(' ', '_')] = {
        'avg_specialization': float(scores.mean()),
        'specialization_scores': scores.tolist(),
        'expert_domain_dist': dist.tolist(),
    }

with open(output_dir / 'specialization_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {output_dir}/")